In [2]:
import pandas as pd
import numpy as np

In [5]:
customers = pd.read_csv(r"C:\Users\pavan kumar\Downloads\customers.csv")
transactions = pd.read_csv(r"C:\Users\pavan kumar\Downloads\transactions.csv")
high_risk = pd.read_csv(r"C:\Users\pavan kumar\Downloads\high_risk_countries.csv")

In [7]:
# Convert Date column to datetime
transactions['Date'] = pd.to_datetime(transactions['Date'])


In [9]:
# Remove duplicates
customers.drop_duplicates(inplace=True)
transactions.drop_duplicates(inplace=True)

In [11]:
# Handle missing values
customers.fillna({
    'Income': customers['Income'].median(),
    'Occupation': 'Unknown',
    'KYC_Status': 'Incomplete'
}, inplace=True)

transactions.fillna({
    'Amount': transactions['Amount'].median(),
    'Transaction_Type': 'Unknown'
}, inplace=True)

In [12]:
# Standardize text columns
customers['Country'] = customers['Country'].str.strip().str.title()
transactions['Country'] = transactions['Country'].str.strip().str.title()

In [13]:
# FEATURE ENGINEERING

# Total transaction amount per customer
total_amount = transactions.groupby('Customer_ID')['Amount'].sum().reset_index()
total_amount.rename(columns={'Amount': 'Total_Amount'}, inplace=True)


In [14]:
# Transaction frequency per customer
txn_count = transactions.groupby('Customer_ID').size().reset_index(name='Transaction_Count')

In [15]:

# Average transaction amount
avg_amount = transactions.groupby('Customer_ID')['Amount'].mean().reset_index()
avg_amount.rename(columns={'Amount': 'Avg_Amount'}, inplace=True)


In [18]:

# Transactions in high-risk countries
high_risk_list = high_risk['Country'].tolist()
transactions['High_Risk_Flag'] = transactions['Country'].isin(high_risk_list).astype(int)

high_risk_txn = transactions.groupby('Customer_ID')['High_Risk_Flag'].sum().reset_index()
high_risk_txn.rename(columns={'High_Risk_Flag': 'High_Risk_Txn_Count'}, inplace=True)


In [19]:
# Large transactions (> 5 lakh)
transactions['Large_Txn'] = (transactions['Amount'] > 500000).astype(int)
large_txn = transactions.groupby('Customer_ID')['Large_Txn'].sum().reset_index()

In [20]:
# Round number transactions (like 10000, 50000)
transactions['Round_Txn'] = transactions['Amount'] % 10000 == 0
round_txn = transactions.groupby('Customer_ID')['Round_Txn'].sum().reset_index()
round_txn.rename(columns={'Round_Txn': 'Round_Txn_Count'}, inplace=True)

In [21]:
# 4. MERGE FEATURES
# =========================

features = customers.merge(total_amount, on='Customer_ID', how='left') \
                    .merge(txn_count, on='Customer_ID', how='left') \
                    .merge(avg_amount, on='Customer_ID', how='left') \
                    .merge(high_risk_txn, on='Customer_ID', how='left') \
                    .merge(large_txn, on='Customer_ID', how='left') \
                    .merge(round_txn, on='Customer_ID', how='left')

# Fill NaN after merge
features.fillna(0, inplace=True)

In [22]:
# 5. RISK SCORING
# =========================

def risk_score(row):
    score = 0
    
    # High-risk country transactions
    if row['High_Risk_Txn_Count'] > 0:
        score += 30
        
    # Large transactions
    if row['Large_Txn'] > 2:
        score += 25
        
    # High frequency
    if row['Transaction_Count'] > 10:
        score += 20
        
    # Incomplete KYC
    if row['KYC_Status'] == 'Incomplete':
        score += 15
        
    # Round transactions
    if row['Round_Txn_Count'] > 3:
        score += 10
        
    return score
features['Risk_Score'] = features.apply(risk_score, axis=1)

# Risk category
def risk_category(score):
    if score <= 30:
        return "Low"
    elif score <= 60:
        return "Medium"
    else:
        return "High"
features['Risk_Category'] = features['Risk_Score'].apply(risk_category)



In [23]:
# 6. OUTPUT
# =========================

# Top suspicious customers
suspicious = features.sort_values(by='Risk_Score', ascending=False)

In [24]:
# Save output
features.to_csv("aml_features_output.csv", index=False)
suspicious.to_csv("suspicious_customers.csv", index=False)


In [1]:
import os
os.getcwd()

'C:\\Users\\pavan kumar'

In [26]:
print(" AML Feature Engineering Completed!")
print(features.head())

 AML Feature Engineering Completed!
   Customer_ID  Age  Country  Occupation   Income KYC_Status  Total_Amount  \
0         1001   59      Usa    Salaried  1659933   Complete     1656380.0   
1         1002   49      Usa  Unemployed   935716   Complete     2353056.0   
2         1003   35      Usa     Student  1354454   Complete     1773995.0   
3         1004   63      Usa    Salaried  1975846   Complete     1814464.0   
4         1005   28  Nigeria  Unemployed  1173548   Complete     2535386.0   

   Transaction_Count     Avg_Amount  High_Risk_Txn_Count  Large_Txn  \
0                4.0  414095.000000                  0.0        2.0   
1                5.0  470611.200000                  1.0        2.0   
2                4.0  443498.750000                  1.0        2.0   
3                3.0  604821.333333                  1.0        2.0   
4                3.0  845128.666667                  0.0        3.0   

   Round_Txn_Count  Risk_Score Risk_Category  
0              0.0   